# **Preprocessing Head**

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


class Config:

    def __init__(self):
        self.root_dir = "/content/dataset1/"
        self.save_root = "/content/output_head"

        self.split = "train"
        self.region = "head"

        self.save_grayscale = True
        self.normalize_output_condition = True
        self.fail_if_label_missing = True
        self.save_ext = ".png"

        self.detection_threshold = 0.50
        self.model_path = "/content/blaze_face_short_range.tflite"

        self.head_scale = 1.25
        self.head_vertical_shift = 0.05

        self.gray_fill_value = 128

        self.enable_adaptive_smoothing = True
        self.motion_scale = 20.0


class MediaPipeDetector:
    def __init__(self, model_path, threshold=0.5):
        self.threshold = threshold

        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceDetectorOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_detection_confidence=0.2,
        )
        self.detector = vision.FaceDetector.create_from_options(options)

    def detect(self, frame_bgr):
        try:
            if len(frame_bgr.shape) == 2:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
            else:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result = self.detector.detect(mp_image)
        except Exception:
            return None

        if not result.detections:
            return None

        best_face = None
        best_area = -1.0

        for detection in result.detections:
            score = float(detection.categories[0].score) if detection.categories else 0.0
            if score < self.threshold:
                continue

            bbox = detection.bounding_box
            x1 = float(bbox.origin_x)
            y1 = float(bbox.origin_y)
            x2 = float(bbox.origin_x + bbox.width)
            y2 = float(bbox.origin_y + bbox.height)

            area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
            if area > best_area:
                best_area = area
                best_face = {
                    "score": score,
                    "bbox": [x1, y1, x2, y2],
                }

        return best_face

    def close(self):
        self.detector.close()


def standardize_condition_name(condition):
    condition = condition.strip().lower()
    mapping = {
        "nightnoglasses": "nightnoglasses",
        "night_no_glasses": "nightnoglasses",
        "night-noglasses": "nightnoglasses",
        "night_noglasses": "nightnoglasses",
        "noglasses": "noglasses",
        "no_glasses": "noglasses",
        "nightglasses": "nightglasses",
        "glasses": "glasses",
        "sunglasses": "sunglasses",
    }
    return mapping.get(condition, condition)


def build_output_condition(condition, cfg):
    if cfg.normalize_output_condition:
        return standardize_condition_name(condition)
    return condition


def parse_validation_or_test_video_name(video_stem):
    parts = video_stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Invalid filename format: {video_stem}")

    person_id = parts[0]
    state = parts[-1]
    raw_condition = "_".join(parts[1:-1])
    condition = standardize_condition_name(raw_condition)
    return person_id, condition, state


def label_suffix_for_region(region):
    region = region.lower()
    if region != "head":
        raise ValueError("Only head supported")
    return "head"


def load_labels(label_path):
    if not os.path.isfile(label_path):
        raise FileNotFoundError(label_path)

    with open(label_path, "r") as f:
        content = f.read().strip()

    return [int(ch) for ch in content if ch.isdigit()]


def get_video_fps(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 30

    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    if fps is None or fps <= 0:
        return 30

    return int(round(fps))


def get_fps_fallback_limits(fps):
    fps_int = int(round(fps)) if fps and fps > 0 else 30
    if fps_int == 30:
        return 4, 18
    return 2, 9


def compute_head_box_from_face(face, cfg):
    x1, y1, x2, y2 = face["bbox"]

    face_w = x2 - x1
    face_h = y2 - y1

    cx = x1 + face_w / 2.0
    cy = y1 + face_h / 2.0
    cy -= cfg.head_vertical_shift * face_h

    size = int(round(max(face_w, face_h) * cfg.head_scale))
    size = max(size, 1)

    x1_new = int(round(cx - size / 2.0))
    y1_new = int(round(cy - size / 2.0))
    x2_new = x1_new + size
    y2_new = y1_new + size

    return [x1_new, y1_new, x2_new, y2_new]


def smooth_box(prev_box, curr_box, motion_scale=20.0):
    if prev_box is None:
        return [float(v) for v in curr_box]

    prev = np.array(prev_box, dtype=np.float32)
    curr = np.array(curr_box, dtype=np.float32)

    movement = np.linalg.norm(curr - prev)
    motion_factor = min(1.0, movement / motion_scale)

    alpha = 0.7 - 0.4 * motion_factor
    smoothed = alpha * prev + (1.0 - alpha) * curr

    return smoothed.astype(np.float32).tolist()


def crop_with_padding(img, x1, y1, x2, y2):
    h, w = img.shape[:2]

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)

    if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
        img = cv2.copyMakeBorder(
            img,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            borderType=cv2.BORDER_REPLICATE
        )

    x1 += pad_left
    x2 += pad_left
    y1 += pad_top
    y2 += pad_top

    return img[y1:y2, x1:x2]


def extract_crop_from_box(frame_bgr, box_xyxy, save_grayscale):
    if box_xyxy is None:
        return None

    x1, y1, x2, y2 = box_xyxy

    if save_grayscale:
        if len(frame_bgr.shape) == 2:
            img = frame_bgr
        else:
            img = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    else:
        if len(frame_bgr.shape) == 2:
            img = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2BGR)
        else:
            img = frame_bgr

    crop = crop_with_padding(img, x1, y1, x2, y2)

    if crop.size == 0:
        return None

    return crop


def make_gray_from_box(box_xyxy, gray_fill_value, save_grayscale):
    if box_xyxy is not None:
        x1, y1, x2, y2 = box_xyxy
        h = max(1, y2 - y1)
        w = max(1, x2 - x1)
        if save_grayscale:
            return np.full((h, w), gray_fill_value, dtype=np.uint8)
        return np.full((h, w, 3), gray_fill_value, dtype=np.uint8)

    if save_grayscale:
        return np.full((32, 32), gray_fill_value, dtype=np.uint8)
    return np.full((32, 32, 3), gray_fill_value, dtype=np.uint8)


class OnlineHeadCropper:
    def __init__(self, cfg, max_prev_crop, max_prev_position):
        self.cfg = cfg
        self.max_prev_crop = max_prev_crop
        self.max_prev_position = max_prev_position

        self.prev_crop = None
        self.prev_box = None
        self.prev_real_box = None
        self.missing_count = 0

    def step(self, frame_bgr, face):
        if face is not None:
            raw_box = compute_head_box_from_face(face, self.cfg)

            if self.cfg.enable_adaptive_smoothing:
                box_xyxy = smooth_box(
                    self.prev_real_box,
                    raw_box,
                    motion_scale=self.cfg.motion_scale
                )
            else:
                box_xyxy = raw_box

            box_xyxy = [int(round(v)) for v in box_xyxy]

            crop = extract_crop_from_box(
                frame_bgr=frame_bgr,
                box_xyxy=box_xyxy,
                save_grayscale=self.cfg.save_grayscale,
            )

            if crop is not None:
                self.prev_crop = crop
                self.prev_box = box_xyxy
                self.prev_real_box = box_xyxy
                self.missing_count = 0
                return crop, self.prev_box, 1, "current"

        self.missing_count += 1

        if self.prev_crop is not None and self.missing_count <= self.max_prev_crop:
            return self.prev_crop.copy(), self.prev_box, 0, "prev_crop"

        if self.prev_box is not None and self.missing_count <= self.max_prev_position:
            crop = extract_crop_from_box(
                frame_bgr=frame_bgr,
                box_xyxy=self.prev_box,
                save_grayscale=self.cfg.save_grayscale,
            )
            if crop is not None:
                return crop, self.prev_box, 0, "prev_position"

        gray = make_gray_from_box(
            box_xyxy=self.prev_box,
            gray_fill_value=self.cfg.gray_fill_value,
            save_grayscale=self.cfg.save_grayscale,
        )
        return gray, self.prev_box, 0, "gray"


def process_video(video_path, label_path, output_dir, save_prefix, region, detector, cfg):
    labels = load_labels(label_path)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    reported_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = get_video_fps(video_path)
    max_prev_crop, max_prev_position = get_fps_fallback_limits(fps)

    cropper = OnlineHeadCropper(cfg, max_prev_crop, max_prev_position)

    frame_idx = 0
    actual_frames_read = 0

    detected_count = 0
    prev_crop_count = 0
    prev_position_count = 0
    gray_count = 0
    valid_count = 0
    invalid_count = 0

    try:
        while frame_idx < len(labels):
            ret, frame_bgr = cap.read()
            if not ret:
                break

            actual_frames_read += 1

            face = detector.detect(frame_bgr)
            crop, box_xyxy, validity_mask, mode = cropper.step(frame_bgr, face)

            if mode == "current":
                detected_count += 1
            elif mode == "prev_crop":
                prev_crop_count += 1
            elif mode == "prev_position":
                prev_position_count += 1
            elif mode == "gray":
                gray_count += 1
            else:
                raise ValueError(f"Unknown crop mode: {mode}")

            if validity_mask == 1:
                valid_count += 1
            else:
                invalid_count += 1

            label = labels[frame_idx]
            save_name = f"{save_prefix}_{frame_idx:06d}_{validity_mask}_{label}{cfg.save_ext}"
            save_path = os.path.join(output_dir, save_name)

            ok = cv2.imwrite(save_path, crop)
            if not ok:
                raise RuntimeError(f"Failed to save image: {save_path}")

            frame_idx += 1

    finally:
        cap.release()

    total = frame_idx if frame_idx > 0 else 1

    print("=" * 80)
    print(f"Video                : {os.path.basename(video_path)}")
    print(f"Region               : {region}")
    print(f"Output folder        : {output_dir}")
    print(f"FPS                  : {fps}")
    print(f"Prev crop limit      : {max_prev_crop}")
    print(f"Prev position limit  : {max_prev_position}")
    print(f"Reported video frames: {reported_total_frames}")
    print(f"Actual frames read   : {actual_frames_read}")
    print(f"Total labels         : {len(labels)}")
    print(f"Frames saved         : {frame_idx}")
    print(f"Valid frames         : {valid_count}")
    print(f"Invalid frames       : {invalid_count}")
    print("-" * 80)
    print(f"Detected normally    : {detected_count} ({detected_count/total:.2%})")
    print(f"Prev crop reused     : {prev_crop_count} ({prev_crop_count/total:.2%})")
    print(f"Prev position        : {prev_position_count} ({prev_position_count/total:.2%})")
    print(f"Gray fallback        : {gray_count} ({gray_count/total:.2%})")
    print("-" * 80)

    missing_from_video = len(labels) - actual_frames_read
    if missing_from_video > 0:
        print(f"Warning              : video ended early by {missing_from_video} frame(s)")
    elif missing_from_video < 0:
        print(f"Warning              : video has {-missing_from_video} extra frame(s)")
    else:
        print("Status               : labels and readable frames matched")
    print("=" * 80)


def process_train(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name

        for condition_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            raw_condition = condition_entry.name
            out_condition = build_output_condition(raw_condition, cfg)

            for file_entry in sorted(os.scandir(condition_entry.path), key=lambda e: e.name):
                if not file_entry.is_file() or not file_entry.name.lower().endswith(".avi"):
                    continue

                behavior = os.path.splitext(file_entry.name)[0]
                video_path = file_entry.path
                label_path = os.path.join(condition_entry.path, f"{person_id}_{behavior}_{suffix}.txt")

                if not os.path.isfile(label_path):
                    msg = f"Missing train label: {label_path}"
                    if cfg.fail_if_label_missing:
                        raise FileNotFoundError(msg)
                    print(f"[SKIP] {msg}")
                    continue

                output_dir = os.path.join(cfg.save_root, person_id, out_condition, behavior)
                os.makedirs(output_dir, exist_ok=True)

                save_prefix = f"{person_id}_{out_condition}_{behavior}"

                process_video(
                    video_path=video_path,
                    label_path=label_path,
                    output_dir=output_dir,
                    save_prefix=save_prefix,
                    region=cfg.region,
                    detector=detector,
                    cfg=cfg,
                )


def process_validation(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        folder_person_id = person_entry.name

        for file_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not file_entry.is_file() or not file_entry.name.lower().endswith((".avi", ".mp4")):
                continue

            video_stem = os.path.splitext(file_entry.name)[0]
            person_id, condition, state = parse_validation_or_test_video_name(video_stem)

            if person_id != folder_person_id:
                raise ValueError(f"Subject mismatch: {file_entry.path}")

            label_path = os.path.join(person_entry.path, f"{person_id}_{condition}_mixing_{suffix}.txt")

            if not os.path.isfile(label_path):
                msg = f"Missing validation label: {label_path}"
                if cfg.fail_if_label_missing:
                    raise FileNotFoundError(msg)
                print(f"[SKIP] {msg}")
                continue

            out_condition = build_output_condition(condition, cfg)
            output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
            os.makedirs(output_dir, exist_ok=True)
            save_prefix = f"{person_id}_{out_condition}_{state}"

            process_video(
                video_path=file_entry.path,
                label_path=label_path,
                output_dir=output_dir,
                save_prefix=save_prefix,
                region=cfg.region,
                detector=detector,
                cfg=cfg,
            )


def process_test(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    label_root = os.path.join(cfg.root_dir, "test_label_txt", "wh")
    if not os.path.isdir(label_root):
        raise FileNotFoundError(f"Test label folder not found: {label_root}")

    for entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not entry.is_file() or not entry.name.lower().endswith((".avi", ".mp4")):
            continue

        video_stem = os.path.splitext(entry.name)[0]
        person_id, condition, state = parse_validation_or_test_video_name(video_stem)
        label_path = os.path.join(label_root, f"{person_id}_{condition}_mixing_drowsiness.txt")

        if not os.path.isfile(label_path):
            msg = f"Missing test label: {label_path}"
            if cfg.fail_if_label_missing:
                raise FileNotFoundError(msg)
            print(f"[SKIP] {msg}")
            continue

        out_condition = build_output_condition(condition, cfg)
        output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
        os.makedirs(output_dir, exist_ok=True)
        save_prefix = f"{person_id}_{out_condition}_{state}"

        process_video(
            video_path=entry.path,
            label_path=label_path,
            output_dir=output_dir,
            save_prefix=save_prefix,
            region=cfg.region,
            detector=detector,
            cfg=cfg,
        )


def run(cfg):
    os.makedirs(cfg.save_root, exist_ok=True)

    detector = MediaPipeDetector(
        model_path=cfg.model_path,
        threshold=cfg.detection_threshold,
    )

    try:
        split = cfg.split.lower()
        if split == "train":
            process_train(cfg, detector)
        elif split in {"validation", "val"}:
            process_validation(cfg, detector)
        elif split == "test":
            process_test(cfg, detector)
        else:
            raise ValueError(f"Unsupported split: {cfg.split}")
    finally:
        detector.close()


if __name__ == "__main__":
    cfg = Config()

    cfg.root_dir = "/content/dataset1/"
    cfg.save_root = "/content/output_head"

    cfg.split = "train"
    cfg.region = "head"

    cfg.detection_threshold = 0.50
    cfg.model_path = "/content/blaze_face_short_range.tflite"

    cfg.head_scale = 1.25
    cfg.head_vertical_shift = 0.05

    cfg.enable_adaptive_smoothing = True
    cfg.motion_scale = 20.0

    cfg.save_grayscale = True
    cfg.gray_fill_value = 128
    cfg.fail_if_label_missing = True
    cfg.normalize_output_condition = True

    run(cfg)